In [31]:
#@title Setup: Install and Import Libraries
!pip install pandas geopandas fiona gerrychain -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 41.5 MB/s eta 0:00:00


In [33]:
#@title Initializations and Path Declarations
import os
from google.colab import drive
from pathlib import Path

import json
import ast
from typing import Dict, List, Any

import geopandas as gpd
from gerrychain import Graph
from tqdm.notebook import tqdm

# Mount Google Drive
drive.mount('/content/drive')
GOOGLE_DRIVE_DATA_DIR = "/content/drive/MyDrive/project-data/study__ca-stv"

# Define the destination path in the Colab environment
COLAB_DATA_DIR = "/content/ca-stv"

if not os.path.exists(COLAB_DATA_DIR):
    os.symlink(GOOGLE_DRIVE_DATA_DIR, COLAB_DATA_DIR)
    print(f"Symbolic link created from '{GOOGLE_DRIVE_DATA_DIR}' to '{COLAB_DATA_DIR}'")
else:
    print(f"Directory or symbolic link already exists at '{COLAB_DATA_DIR}'")

# Update the paths in your configurations to use the mounted directory
GPKG_PATH = Path(os.path.join(COLAB_DATA_DIR, "CA_data", "ca_districtr_bg_view_v1.gpkg"))
PLANS_DIR = Path(os.path.join(COLAB_DATA_DIR, "districting"))
OUTPUT_DIR = Path("./vk_run_settings") # Keep output in the default output directory



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directory or symbolic link already exists at '/content/ca-stv'
Please ensure your .gpkg file is in '/content/ca-stv/CA_data/'
And your .json plan files are in '/content/ca-stv/districting/'


In [37]:
#@title Parameter Setting
GPKG_LAYER = "ca_districtr_bg_view_v1"
DISTRICT_NUMS_TO_PROCESS = [8] # district numbers to process
PLANS_TO_PROCESS_PER_FILE = 10 # number of district plans to process from the start of each file
NUM_CANDIDATES = 5 # number of candidates in slate
ALPHA_SWEEP_VALUES = [1.0] # [0.6, 0.7, 0.8, 0.9]

In [38]:
#@title Helper functions
def load_graph_from_gpkg(gpkg_path: Path, layer: str) -> Graph:
    """
    Loads a gerrychain Graph from a specified layer in a GeoPackage file.

    Args:
        gpkg_path: The file path to the GeoPackage.
        layer: The name of the layer to load from the GeoPackage.

    Returns:
        A gerrychain Graph object with node data populated from the file.
    """
    print(f"Reading GeoPackage layer '{layer}' from {gpkg_path}...")
    if not gpkg_path.exists():
        raise FileNotFoundError(f"GeoPackage file not found at: {gpkg_path}")

    gdf = gpd.read_file(gpkg_path, layer=layer)
    print("Creating graph from GeoDataFrame...")
    graph = Graph.from_geodataframe(gdf)
    print(f"Graph created successfully with {len(graph.nodes)} nodes.")
    return graph


def calculate_voter_proportions(
    plan_assignment_list: List[int], graph: Graph, district_num: int
) -> Dict[str, List[float]]:
    """
    Calculates the proportion of Hispanic and Non-Hispanic voters in each district.

    Args:
        plan_assignment_list: A list where the index is the block ID and the
                              value is the assigned district ID.
        graph: The gerrychain Graph containing population data for each block.
        district_num: The total number of districts in the plan.

    Returns:
        A dictionary with the proportional representation of each voter bloc.
    """
    # Initialize population counters: {district_id: [hispanic_vap, non_hispanic_vap]}
    district_pops = {i: [0, 0] for i in range(district_num)}

    # Aggregate population data by iterating through the graph nodes
    for block_idx, district_idx in enumerate(plan_assignment_list):
        node_data = graph.nodes[block_idx]
        hisp_vap = node_data.get('hvap_20', 0)
        total_vap = node_data.get('total_vap_20', 0)

        district_pops[district_idx][0] += hisp_vap
        district_pops[district_idx][1] += (total_vap - hisp_vap)

    # Normalize to get proportions
    bloc_voter_props = {"H": [], "NH": []}
    for i in range(district_num):
        hisp_pop, non_hisp_pop = district_pops[i]
        total_district_pop = hisp_pop + non_hisp_pop

        if total_district_pop > 0:
            bloc_voter_props["H"].append(hisp_pop / total_district_pop)
            bloc_voter_props["NH"].append(non_hisp_pop / total_district_pop)
        else:
            # Handle empty districts to avoid division by zero
            bloc_voter_props["H"].append(0.0)
            bloc_voter_props["NH"].append(0.0)

    return bloc_voter_props


def generate_settings_file(
    district_num: int,
    plan_index: int,
    voter_props: Dict[str, List[float]],
    output_dir: Path,
    alpha_val: float
):
    """
    Constructs the settings dictionary and saves it as a JSON file.

    Args:
        district_num: The total number of districts.
        plan_index: The index of the current districting plan.
        voter_props: The calculated voter proportions for the plan.
        output_dir: The directory to save the output file in.
        alpha_val: The alpha value to use for voter cohesion.
    """
    output_settings = {
        "bloc_voter_props": voter_props,
        "cohesion_parameters": {
            "H": {"H": 0.77, "NH": 0.23},
            "NH": {"H": 0.52, "NH": 0.48},
        },
        "alphas": {
            "H": {"H": alpha_val, "NH": alpha_val},
            "NH": {"H": alpha_val, "NH": alpha_val}
        },
        "slate_to_candidates": {
            "H": [f"H{i+1}" for i in range(NUM_CANDIDATES)],
            "NH": [f"NH{i+1}" for i in range(NUM_CANDIDATES)],
        },
    }

    output_dir.mkdir(parents=True, exist_ok=True)

    alpha_str = str(alpha_val).replace('.', 'p')
    file_name = f"wc_{district_num}d_map_{plan_index}_alpha_{alpha_str}.json"
    output_path = output_dir / file_name

    with open(output_path, "w") as f:
        json.dump(output_settings, f, indent=4)

In [40]:
#@title Run main execution
def main():
    """Main function to orchestrate the settings file generation."""
    try:
        graph = load_graph_from_gpkg(GPKG_PATH, GPKG_LAYER)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure your GeoPackage file is uploaded to the correct directory.")
        return

    for district_num in DISTRICT_NUMS_TO_PROCESS:
        plan_file_path = PLANS_DIR / f"chain_out/ca_{district_num}_dist.json"

        if not plan_file_path.exists():
            print(f"\n Warning: Plan file not found at {plan_file_path}. Skipping.")
            continue

        print(f"\nProcessing plans for {district_num} districts from {plan_file_path.name}...")

        with open(plan_file_path) as f:
            lines = f.readlines()

        plans_to_run = lines[:PLANS_TO_PROCESS_PER_FILE]

        for i, line in enumerate(tqdm(plans_to_run, desc=f"Generating for {district_num} districts")):
            try:
                # ast.literal_eval is used to safely evaluate the string as a Python literal
                plan_data = ast.literal_eval(line.strip())
                plan_assignment = plan_data['assignment']
            except (SyntaxError, KeyError, ValueError) as e:
                print(f"Error processing line {i+1} in {plan_file_path}: {e}. Skipping.")
                continue

            voter_props = calculate_voter_proportions(plan_assignment, graph, district_num)
            for alpha in ALPHA_SWEEP_VALUES:
                print(f"Generating settings file for district {i} with alpha {alpha}")
                generate_settings_file(
                    district_num=district_num,
                    plan_index=i,
                    voter_props=voter_props,
                    output_dir=OUTPUT_DIR,
                    alpha_val=alpha
                )
            # generate_settings_file(district_num, i, voter_props, OUTPUT_DIR)

    print("\n Completed without fail")


if __name__ == "__main__":
    main()

Reading GeoPackage layer 'ca_districtr_bg_view_v1' from /content/ca-stv/CA_data/ca_districtr_bg_view_v1.gpkg...
Creating graph from GeoDataFrame...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gerrychain/graph/graph.py:406: UserWarning: Found islands (degree-0 nodes). Indices of islands: {20398, 9431}
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gerrychain/graph/graph.py:266: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  areas = df.geometry.area.to_dict()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent date

Graph created successfully with 25607 nodes.

Processing plans for 8 districts from ca_8_dist.json...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Generating for 8 districts:   0%|          | 0/10 [00:00<?, ?it/s]

Generating settings file for district 0 with alpha 0.6
Generating settings file for district 0 with alpha 0.7
Generating settings file for district 0 with alpha 0.8
Generating settings file for district 0 with alpha 0.9
Generating settings file for district 1 with alpha 0.6
Generating settings file for district 1 with alpha 0.7
Generating settings file for district 1 with alpha 0.8
Generating settings file for district 1 with alpha 0.9
Generating settings file for district 2 with alpha 0.6
Generating settings file for district 2 with alpha 0.7
Generating settings file for district 2 with alpha 0.8
Generating settings file for district 2 with alpha 0.9
Generating settings file for district 3 with alpha 0.6
Generating settings file for district 3 with alpha 0.7
Generating settings file for district 3 with alpha 0.8
Generating settings file for district 3 with alpha 0.9
Generating settings file for district 4 with alpha 0.6
Generating settings file for district 4 with alpha 0.7
Generating

In [36]:
#@title Sanity check
!ls -lh vk_run_settings
print("\n--- Contents of one of the generated files ---")
!cat vk_run_settings/wc_8d_map_0.json

total 40K
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_0.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_1.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_2.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_3.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_4.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_5.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_6.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_7.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_8.json
-rw-r--r-- 1 root root 1.2K Oct 14 18:31 wc_8d_map_9.json

--- Contents of one of the generated files ---
{
    "bloc_voter_props": {
        "H": [
            0.49364955380736775,
            0.20521498947034064,
            0.3427755447589663,
            0.2207558049947498,
            0.3576923337909882,
            0.40406196541599126,
            0.4463053617401402,
            0.36532489769067195
        ],
        "NH": [
            0.5063504461926323,
 